# 04 HAR-RV Multi-Horizon Modeling

Heterogeneous Autoregressive Realized Volatility model with 3-horizon aggregation.

In [ ]:
import sys
sys.path.insert(0, '../')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.volatility_forecasting.data.loader import download_data
from src.volatility_forecasting.data.preprocessor import DataPreprocessor
from src.volatility_forecasting.models.har_model import (
    compute_realized_volatility_proxy,
    build_har_features,
    fit_har_rv
)
from src.volatility_forecasting.config import DATA_START_DATE, DATA_END_DATE, TICKER, HAR_LAGS_WEEKLY, HAR_LAGS_MONTHLY
from src.volatility_forecasting.logger import setup_logger

logger = setup_logger('notebook')
plt.style.use('seaborn-v0_8-darkgrid')

In [ ]:
# Load data
price_data = download_data(ticker=TICKER, start=DATA_START_DATE, end=DATA_END_DATE)
preprocessor = DataPreprocessor(price_data)
returns = preprocessor.compute_log_returns().dropna()

print(f"Data loaded: {len(returns)} observations")

In [ ]:
# Compute realized volatility proxy
rv_proxy = compute_realized_volatility_proxy(returns)

print(f"RV proxy computed: {len(rv_proxy)} observations")
print(f"Mean RV: {rv_proxy.mean():.6f}")
print(f"Std RV: {rv_proxy.std():.6f}")

In [ ]:
# Build HAR features
har_features = build_har_features(
    rv_proxy,
    lags_weekly=HAR_LAGS_WEEKLY,
    lags_monthly=HAR_LAGS_MONTHLY
)

print(f"HAR features built: {len(har_features)} observations")
print("Columns:", har_features.columns.tolist())
print("\nFirst rows:")
print(har_features.head())

In [ ]:
# Fit HAR-RV model
har_model = fit_har_rv(returns)

# Display results
print("HAR-RV Model Summary:")
print(har_model.summary())

# Extract coefficients
print("\nCoefficients:")
print(f"Intercept: {har_model.params['const']:.6f}")
print(f"Daily (RV_d): {har_model.params['RV_d']:.6f}")
print(f"Weekly (RV_w): {har_model.params['RV_w']:.6f}")
print(f"Monthly (RV_m): {har_model.params['RV_m']:.6f}")

In [ ]:
# Residual diagnostics
residuals = har_model.resid

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Residuals over time
axes[0, 0].plot(residuals)
axes[0, 0].set_title('HAR-RV Residuals')
axes[0, 0].set_ylabel('Residual')

# Distribution
axes[0, 1].hist(residuals, bins=50, edgecolor='black')
axes[0, 1].set_title('Residual Distribution')
axes[0, 1].set_xlabel('Residual')

# Q-Q plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title('Q-Q Plot')

# ACF
axes[1, 1].bar(range(1, 21), [residuals.autocorr(lag=i) for i in range(1, 21)])
axes[1, 1].set_title('Autocorrelation of Residuals')
axes[1, 1].set_xlabel('Lag')

plt.tight_layout()
plt.savefig('../report/figures/04_har_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

logger.info("HAR-RV modeling complete")